### Citi Bike Prediction Project — Feature Engineering

his notebook prepares the Citi Bike station-level dataset for predictive modeling. The workflow defines the target variable, selects the modeling features, applies preprocessing transformations, and exports the processed training and testing datasets for use in the next notebook.

Because the project is focused on forecasting future low dock availability, the dataset is sorted chronologically before splitting so model evaluation better reflects a real-world prediction setting. The final output of this notebook is a reusable preprocessing pipeline along with modeling-ready train and test datasets.

#### Load libraries

This section imports the Python libraries used for feature engineering, preprocessing, and artifact export. These tools support train/test splitting, variable transformation, and saving reusable preprocessing objects for the modeling stage.

In [1]:
# Core analysis libraries
import pandas as pd
import numpy as np
import datetime
import time

# File path locations
from pathlib import Path

# Load data pull libraries
import requests
import json

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Improve plot styling
sns.set()

# Show all columns
pd.set_option('display.max_columns', None)

#### File directories

This section defines the project paths used to load the feature-engineered dataset and save modeling artifacts. Keeping file paths centralized makes the notebook easier to maintain and supports a more reproducible project structure.

In [2]:
# File paths
PROJECT_ROOT = Path.cwd().resolve().parent
CLEAN_DATA_DIR = PROJECT_ROOT / "data/clean_data"
RAW_DATA_DIR = PROJECT_ROOT / "data/raw_data"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"

#### Import and inspect data

This section loads the prepared Citi Bike dataset and performs a quick inspection of its structure. The goal is to confirm that expected variables are present, data types look appropriate, and the dataset is ready for feature engineering and splitting.

In [3]:
# Load data saved as a parquet file
data = pd.read_parquet(CLEAN_DATA_DIR / "01_citi_bike_prediction_preprocessing.parquet")

In [4]:
# Number of rows and columns
data.shape

(717536, 17)

In [5]:
# Show first five records of data
data.head(5)

,station_id,num_bikes_available,num_docks_available,snapshot_time,snapshot_hr,snapshot_weekday,citi_bike_station_name,citi_bike_lat,citi_bike_lon,capacity,meters_to_nearest_mta_station,current_dock_pct,future_num_docks_available,future_snapshot_time,future_dock_pct,minutes_to_future,lda_30min
0,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 04:07:35,4,6,28 Ave & 44 St,40.764089,-73.910651,19.0,893.80959,0.421053,6.0,2026-03-22 04:37:35,0.315789,30.0,0
1,00284700-9d22-42ce-8485-113fed9879c1,12,6,2026-03-22 04:37:35,4,6,28 Ave & 44 St,40.764089,-73.910651,19.0,893.80959,0.315789,8.0,2026-03-22 05:07:35,0.421053,30.0,0
2,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 05:07:35,5,6,28 Ave & 44 St,40.764089,-73.910651,19.0,893.80959,0.421053,8.0,2026-03-22 05:37:35,0.421053,30.0,0
3,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 05:37:35,5,6,28 Ave & 44 St,40.764089,-73.910651,19.0,893.80959,0.421053,8.0,2026-03-22 06:07:35,0.421053,30.0,0
4,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 06:07:35,6,6,28 Ave & 44 St,40.764089,-73.910651,19.0,893.80959,0.421053,8.0,2026-03-22 06:37:35,0.421053,30.0,0


In [6]:
# Data types along with count of missing
data_types_and_counts = pd.DataFrame()
data_types_and_counts["Data Type"] = data.dtypes
data_types_and_counts["Count of Nulls"] = data.isna().sum()
data_types_and_counts

,Data Type,Count of Nulls
station_id,object,0
num_bikes_available,int64,0
num_docks_available,int64,0
snapshot_time,datetime64[ns],0
snapshot_hr,int32,0
snapshot_weekday,int32,0
citi_bike_station_name,object,0
citi_bike_lat,float64,0
citi_bike_lon,float64,0
capacity,float64,0


#### Sort entire dataset by `snapshot_time`

Because this project is forecasting future low dock availability, observations should be ordered chronologically before splitting into training and testing sets. Sorting by `snapshot_time` helps create a more realistic evaluation setup by ensuring the model is trained on earlier observations and tested on later ones.

In [7]:
model_data = data.sort_values(by="snapshot_time", ascending=True).reset_index(drop=True).copy()

#### Target selection

This section defines the prediction target for the modeling workflow. The target variable, `lda_30min`, indicates whether a station is expected to experience low dock availability within the next 30 minutes.

In [8]:
target = "lda_30min"

#### Feature selection

This section identifies the predictor variables used for modeling and groups them by type. Continuous, categorical, and time-based features are separated so they can be transformed appropriately in the preprocessing pipeline.

In [9]:
# Continuous features
continuous_features = [
	"citi_bike_lat", 
	"citi_bike_lon", 
	"capacity", 
	"meters_to_nearest_mta_station"
]

In [10]:
# Categorical features
categorical_features = [
	"station_id"
]

In [11]:
# Time features
time_features = [
	"snapshot_hr", 
	"snapshot_weekday"
]

In [19]:
# Combine model features
model_features = continuous_features + time_features

#### Train and test splits

This section splits the dataset into training and testing subsets using chronological order rather than a random split. This approach better reflects a real-world forecasting scenario, where the model learns from past observations and is evaluated on future ones.

In [20]:
# Training and test sizes
train_size = int(len(model_data) * 0.8)

train = model_data.iloc[:train_size].copy()
test = model_data.iloc[train_size:].copy()

In [21]:
# Training data
X_train = train[model_features]
y_train = train[target]

In [22]:
# Test data
X_test = test[model_features]
y_test = test[target]

#### Build feature processor

This section constructs the preprocessing pipeline used to prepare features for modeling. Numerical variables are scaled, categorical variables are one-hot encoded, and time-based variables are transformed into cyclical representations so that temporal patterns can be captured more effectively.

In [23]:
# Import relevant libraries
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from feature_engine.creation import CyclicalFeatures

In [24]:
# Create preprocessor object to automate tranformations
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), continuous_features),
        #("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("cyclical", CyclicalFeatures(variables=time_features, drop_original=True), time_features)
    ],
    verbose_feature_names_out=False
)

#### Fit and transform data

This section fits the preprocessing pipeline on the training data and applies the learned transformations to both the training and testing sets. Fitting only on the training data helps prevent data leakage and preserves a realistic modeling workflow.

In [25]:
# Fit and transform training data
X_train_processed = preprocessor.fit_transform(X_train)
X_train_processed.shape

(574028, 8)

In [26]:
# Transform to testing data
X_test_processed = preprocessor.transform(X_test)
X_test_processed.shape

(143508, 8)

#### Recreate dataframes

After transformation, this section converts the processed feature arrays back into pandas DataFrames with readable column names. Rebuilding DataFrames makes the transformed data easier to inspect, validate, export, and use in the modeling notebook.

In [27]:
feature_names = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

#### Export for modeling

This section saves the processed training and testing datasets so they can be used directly in the modeling stage. Exporting these files helps separate feature engineering from model development and supports a cleaner CRISP-DM workflow.

In [28]:
X_train_df.to_parquet(CLEAN_DATA_DIR / "03_X_train_processed.parquet")
X_test_df.to_parquet(CLEAN_DATA_DIR / "03_X_test_processed.parquet")

In [29]:
y_train.to_frame().to_parquet(CLEAN_DATA_DIR / "03_y_train.parquet")
y_test.to_frame().to_parquet(CLEAN_DATA_DIR / "03_y_test.parquet")

#### Save to job library

This section saves the fitted preprocessing object as a reusable artifact. Persisting the preprocessor makes it possible to apply the exact same feature transformations during modeling, evaluation, and future inference.

In [30]:
import joblib

joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

['C:\\Users\\jac67\\Documents\\Data and Analytics\\Python\\citi-bike-prediction\\models\\preprocessor.joblib']